In [1]:
from pathlib import Path
import faiss
import numpy as np


# ============================================================
# Configuration
# ============================================================

INDEX_FILE = Path(
    r"C:\Users\User\Nasa Rag SYS\rag\vector_store\processed\vector_store\faiss.index"
)


# ============================================================
# FAISS Vector Store
# ============================================================

class FAISSVectorStore:

    def __init__(self, index_path=INDEX_FILE):

        self.index_path = Path(
            index_path
        )

        self.index = None

        self.load()


    # ========================================================
    # Load FAISS index
    # ========================================================

    def load(self):

        if not self.index_path.exists():

            raise FileNotFoundError(
                f"FAISS index not found: "
                f"{self.index_path}"
            )

        self.index = faiss.read_index(
            str(self.index_path)
        )

        print(
            f"FAISS index loaded."
        )

        print(
            f"Vectors: {self.index.ntotal}"
        )

        print(
            f"Dimension: {self.index.d}"
        )


    # ========================================================
    # Search
    # ========================================================

    def search(
        self,
        query_embedding,
        top_k=5
    ):

        # Make sure shape is:
        # (1, embedding_dimension)

        query_embedding = np.asarray(
            query_embedding,
            dtype=np.float32
        )

        if query_embedding.ndim == 1:

            query_embedding = (
                query_embedding.reshape(1, -1)
            )

        # Search
        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        results = []

        for score, index in zip(
            scores[0],
            indices[0]
        ):

            results.append({
                "chunk_id": int(index),
                "score": float(score)
            })

        return results



In [2]:
from pathlib import Path

INDEX_FILE = Path(
    r"C:\Users\User\Nasa Rag SYS\rag\vector_store\processed\vector_store\faiss.index"
)

print("Exists:", INDEX_FILE.exists())
print("Path:", INDEX_FILE)

Exists: True
Path: C:\Users\User\Nasa Rag SYS\rag\vector_store\processed\vector_store\faiss.index


In [3]:
from sentence_transformers import SentenceTransformer


# ============================================================
# Load BGE embedding model
# ============================================================

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


# ============================================================
# Load FAISS Vector Store
# ============================================================

vector_store = FAISSVectorStore()


# ============================================================
# Test Query
# ============================================================

query = (
    "How does oxygen concentration affect "
    "flame behavior in microgravity?"
)

print("\nQuery:")
print(query)


# ============================================================
# Create Query Embedding
# ============================================================

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("\nQuery embedding shape:")
print(query_embedding.shape)


# ============================================================
# FAISS Search
# ============================================================

results = vector_store.search(
    query_embedding,
    top_k=5
)


# ============================================================
# Display Results
# ============================================================

print("\n================================")
print("FAISS SEARCH RESULTS")
print("================================")

for rank, result in enumerate(
    results,
    start=1
):

    print(
        f"Rank {rank} | "
        f"Chunk ID: {result['chunk_id']} | "
        f"Score: {result['score']:.4f}"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FAISS index loaded.
Vectors: 1200
Dimension: 384

Query:
How does oxygen concentration affect flame behavior in microgravity?

Query embedding shape:
(384,)

FAISS SEARCH RESULTS
Rank 1 | Chunk ID: 243 | Score: 0.7075
Rank 2 | Chunk ID: 175 | Score: 0.7068
Rank 3 | Chunk ID: 289 | Score: 0.7043
Rank 4 | Chunk ID: 241 | Score: 0.7038
Rank 5 | Chunk ID: 205 | Score: 0.7028


In [4]:
import os
import sys

print("Current working directory:")
print(os.getcwd())

print("\nPython path:")
for path in sys.path:
    print(path)

Current working directory:
C:\Users\User\Nasa Rag SYS\rag\vector_store

Python path:
C:\ProgramData\anaconda3\python313.zip
C:\ProgramData\anaconda3\DLLs
C:\ProgramData\anaconda3\Lib
C:\ProgramData\anaconda3

C:\Users\User\AppData\Roaming\Python\Python313\site-packages
C:\ProgramData\anaconda3\Lib\site-packages
C:\ProgramData\anaconda3\Lib\site-packages\win32
C:\ProgramData\anaconda3\Lib\site-packages\win32\lib
C:\ProgramData\anaconda3\Lib\site-packages\Pythonwin
C:\ProgramData\anaconda3\Lib\site-packages\setuptools\_vendor


In [5]:
import sys

VECTOR_STORE_DIR = r"C:\Users\User\Nasa Rag SYS\rag\vector_store"

if VECTOR_STORE_DIR not in sys.path:
    sys.path.insert(0, VECTOR_STORE_DIR)

print("Vector store directory added.")

Vector store directory added.


In [6]:
from Metadata_filter import MetadataFilter

print("MetadataFilter imported successfully!")

MetadataFilter imported successfully!


In [7]:
from pathlib import Path

chunks_file = Path(
    r"C:\Users\User\Nasa Rag SYS\rag\embeddings\processed\chunks\chunks.json"
)

metadata_filter = MetadataFilter(
    chunks_file
)

enriched_results = metadata_filter.enrich_results(
    results
)

Loaded 1200 chunks.


In [8]:
for rank, result in enumerate(enriched_results, start=1):

    print("\n" + "=" * 70)
    print(f"Rank: {rank}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Score: {result['score']:.4f}")

    print("\nMetadata:")
    print(result["metadata"])

    print("\nText:")
    print(result["text"])


Rank: 1
Chunk ID: 243
Score: 0.7075

Metadata:
{'experiment_id': 'EXP-0061', 'source_investigation': 'BASS-II', 'material': 'Nomex', 'material_type': 'solid', 'flow_configuration': 'opposed', 'oxygen_concentration_pct': 15.27, 'pressure_kpa': 68.6, 'flow_velocity_cm_s': 25.84, 'ambient_temperature_c': 23.7, 'outcome': 'self_extinguished', 'risk_level': 'medium', 'test_date': '2026-02-18', 'chunk_type': 'risk_summary', 'source': 'EXP-0061.md'}

Text:
# Experiment EXP-0061

## Fire Risk

- **Risk Score:** 24.7
- **Risk Level:** medium

## Experimental Summary

Experiment EXP-0061 was conducted as part of the
BASS-II investigation using Nomex.

The experiment used an oxygen concentration of
15.27%, a pressure of
68.6 kPa, and a flow velocity of
25.84 cm/s.

The observed flame spread rate was
1.299 mm/s, with a heat release rate of
1.258 kW.

The observed outcome was **self_extinguished**, with a calculated
risk score of 24.7 and a **medium**
risk level.

Rank: 2
Chunk ID: 175
Score: 0.70